# Practical excercise: BERT based persona judge

## Background

We are cooperating with NVIDIA and Valicon to produce a Slovene Nemotron Personas dataset. The dataset will contain synthetic Slovene persona description. It will be useful for various tasks such as market analysis and generating task-specific training data for models based on the interaction with a given persona.

In the first stage we generate demographic attributes based on the actual statistics for Slovenia. Next, we add the personality traits from the OCEAN Big Five specification. Based on these data, an LLM has to generate the general persona description. Specifically it has to generate the description for the following fields:
- `cultural_background`
- `skills_and_expertise`
- `career_goals_and_ambitions`
- `hobbies_and_interests`

However, the LLMs that generate these descriptions are prone to hallucinations and language errors. In this case, we denote with hallucinations part of the generated text that is inconsistent with the input data (wrong place to live, occupation, education level, etc.) or that is wrong in general (generating wrong factual information).

To prevent such errors, we implemented a LLM-as-a-judge approach, where LLM checks the generated personas and accepts or rejects them based on a certain criteria. To perform the scoring, the judge is first asked to find spans of text that include one of the following:
- **contradiction** — states something the record says is false (e.g. claims a job when the record says unemployed)
- **invention** — adds biographical detail that isn't in the record, but isn't contradicted by it either (allowed, even expected)
- **language** — a grammar, agreement, or wording error, unrelated to the record
- **correct** — a case handled well that could easily have gone wrong (e.g. correctly acknowledging retirement instead of inventing a job)
The judge determines the final quality of the generated persona based on the detected spans. If there are major contradictions, generation is rejected. If there are a lot of language errors, the generation is rejected. If the inventions are plaussible and consistent with the persona's bio, the generation gets higher score.

## Your task

You are given 200 Slovene persona generated with 3 different models: GaMS2-27B, Gemma3-27B and DeepSeek-V4-Flash and spans that were marked and labeled by Kimi-K3 judge. Your task is to find out whether a BERT model can be fine-tuned to correctly classify provided spans into the four classes described above.


## 1. Setup

Install `transformers`, `accelerate`, `datasets`, `evaluate`, `scikit-learn` and `pandas`.

In [ ]:
!pip install -q transformers accelerate datasets evaluate scikit-learn pandas

## 2. Get the data

As we work with the Colab, we first need to transfer the data to it. The data lives in the `exercise/data/` folder of the course GitHub repo: https://github.com/zivastebljaj/KCUI-efficient-adaptation-LLMs

Colab starts every session with an empty working directory (`/content`), so you need to bring the data in first. The repo is public, so the easiest way is to clone it straight from a code cell. Prefix a line with `!` to run it as a shell command:

```python
!git clone --depth 1 https://github.com/zivastebljaj/KCUI-efficient-adaptation-LLMs.git
DATA_DIR = "KCUI-efficient-adaptation-LLMs/exercise/data"
```

`--depth 1` downloads only the latest version, without the history. Afterwards, open the **Files** panel (folder icon in the left sidebar) and check that the data is there. You can also run `!ls -R {DATA_DIR}`. You should see:

```text
data/
├── generated_personas/
│   ├── personas_sl_deepseek.jsonl
│   ├── personas_sl_gams2.jsonl
│   └── personas_sl_gemma3.jsonl
└── judge_evaluations/
    ├── kimi_deepseek_analysis.jsonl
    ├── kimi_gams2_analysis.jsonl
    └── kimi_gemma3_analysis.jsonl
```

<details><summary>Alternative: download single files</summary>

Every file in the repo is also available as a raw download at
`https://raw.githubusercontent.com/zivastebljaj/KCUI-efficient-adaptation-LLMs/main/exercise/data/<path>`. Fetch it with `!wget -P <target_dir> <url>`, for example:

```python
!wget -q -P data/generated_personas https://raw.githubusercontent.com/zivastebljaj/KCUI-efficient-adaptation-LLMs/main/exercise/data/generated_personas/personas_sl_gams2.jsonl
```
</details>

<details><summary>Alternative: upload manually</summary>

On GitHub, click **Code → Download ZIP** and unzip it on your computer. Then, in Colab, drag the files from `exercise/data/` into the **Files** panel, or click its upload icon. Keep the folder structure shown above.
</details>

**Note:** Everything in `/content` is deleted when the Colab runtime disconnects or restarts. If that happens, run this step again.

## 3. Build the dataset

### The files

All files are JSON Lines (one JSON object per line).

**Personas**, `personas_sl_{generator}_multiple.jsonl`, hold one record per generation, identified by `(uuid, generation_index)`. The fields you'll need:

| field | meaning |
|---|---|
| `first_name`, `last_name`, `sex`, `age` | basic demographics |
| `marital_status`, `household_status` | family situation |
| `education_level`, `bachelors_field` | education (`bachelors_field` contains `"brez diplome"` if there is no degree) |
| `activity_status` | work status (employed, retired, unemployed, …) |
| `occupation`, `detailed_occupation` | job; may be empty |
| `statistical_region`, `municipality`, `post_code`, `post_town`, `settlement` | location |
| `openness`, `conscientiousness`, `extraversion`, `agreeableness`, `neuroticism` | Big Five traits, each a dict with a `label` key |

**Judge verdicts**, `judge_output/kimi_{generator}_analysis.jsonl`, hold one verdict per generation, with `uuid`, `generation_index` and a list of `spans`. Each span has `text`, `block`, `mark` (this is the label), `covers` and `field`.

### What to build

A table with **one row per flagged span**, containing the span text, the person's record as context, and the mark. Rules:

- skip spans that stand for a whole bullet list (`covers == "bullet_list"`), since those are list fragments rather than sentences
- attach the same context to every span of a generation, regardless of its mark: the person's record fields plus their Big Five profile, turned into a single string
- **don't** use the judge's own per-span `field` annotation as context — the rubric only ever fills that in for `contradiction` spans, so using it as a model input would leak the label
- skip verdicts with no matching persona record, and drop duplicate spans

<details><summary>Hint: matching verdicts to personas</summary>

For each generator, build a dict from `(uuid, generation_index)` to the persona record, then look up each verdict in it.
</details>

<details><summary>Hint: the context string</summary>

Keep it simple and readable, for example:
`Ime: Nada Hren | Spol: ženski | Starost: 67 | ... | Poklic: ... | OCEAN: openness=visoko, ...`

Leave out empty fields like a missing occupation, rather than writing `None`.
</details>

<details><summary>Hint: columns to keep</summary>

Besides the text, context and label, keep `uuid`, because you'll need it for the split.
</details>

Check what you built: how many rows do you have, and how are the labels distributed? Read a few rows and make sure the context makes sense.

## 4. Train/validation split

Make an 80/20 split **by person (`uuid`), not by row**. Each person's record appears in many rows, across 3 generators and several spans each. A random split would put the same people in both train and validation. 

Before splitting:
- drop classes with fewer than 5 examples
- map the labels to integer ids

<details><summary>Hint</summary>

`sklearn.model_selection.GroupShuffleSplit(n_splits=1, test_size=0.2)` with `groups=df["uuid"]`. Afterwards, check that no uuid appears in both splits.
</details>

## 5. Train

Finetune `EMBEDDIA/sloberta` for sequence classification. Each input is the context string, then `[SEP]`, then the span text.

Suggested settings: `max_length=256`, 3 epochs, learning rate `2e-5`, batch size 16, weight decay 0.01, 50 warmup steps, a cosine schedule with `min_lr=2e-6`, evaluation once per epoch, bf16. Track accuracy and macro-F1.

<details><summary>Hint: the pieces you need</summary>

`AutoTokenizer`, `AutoModelForSequenceClassification`, `datasets.Dataset.from_dict`, `DataCollatorWithPadding`, `TrainingArguments`, `Trainer`. Name the label column `labels`, since that's what the Trainer expects.
</details>

<details><summary>Hint: reproducibility</summary>

Call `transformers.set_seed(...)` right before you load the model, so the random weights of the classification head are the same on every run.
</details>

## 6. Evaluate

Print a per-class classification report and a confusion matrix for the validation set. Which marks are easy, and which get confused with each other?

## Bonus

- Replace `GroupShuffleSplit` with `StratifiedGroupKFold`, which keeps the split grouped by person and also balances the classes. Does it change the scores for the rare classes?